# Population feature engineering

This notebook translates the single-axon feature representation into one integrated population signal per trial. Axons are processed separately only to form their variable-duration 10 ms burst envelopes; the observer never receives axon-specific features.

The 1 ms trace contains population spike counts. The 10 ms trace contains the number of axons occupying each burst bin. Its binary support defines population burst events and `silence_lv`, preserving burst duration while excluding simultaneous-axon magnitude from the timing calculation. The manuscript-facing labels **LV_ISI** and **LV_IBI** refer to the 1 ms and 10 ms `silence_lv` representations.

The analysis window begins after subtracting the 100 ms MRG initialization and extends to 2,130 ms. This contains the nominal 2,000 ms stimulus, the latest audited input pulse at 2,020.65 ms, and the full 100 ms simulated response tail. The extractor fails rather than silently trimming if a future response reaches the endpoint. A trace that fills the complete window has two equal zero-duration edge silences; by convention its `silence_lv` is zero and an explicit degeneracy flag is retained.

The primary burst-envelope display is binary population timing support. Concurrent-axon count and recruited-axon fraction are optional rate/intensity views of the same underlying population envelope.

Rate evidence uses endpoint-gated hard-JND votes. Each matched repetition/feature can vote only when its CV0 and CV1 templates are themselves JND-separable. Directly averaging these gated votes retains endpoint reliability as the evidence scale and bounds every intermediate CV within that scale; there is no endpoint-affine or CV-dependent reliability transform. Timing evidence is the bounded signed CV0/CV1 template-distance contrast, accompanied by endpoint-separation diagnostics. Negative and positive values favor CV0 and CV1, respectively.

CV0 uses one generated exact-periodic representation across all 100 historical repetition labels; `rep001_slack` at CV0 is only a compatibility label. Regimes remain in the long-form evidence table for provenance, while a frequency-wide response-equivalence map supplies canonical combined/retained labels and prevents the observer from fitting duplicate responses independently. Equivalence requires all five raw engineered population features to match across every CV and repetition; amplitude proximity and equality only after evidence bounding are insufficient.


In [ ]:
# Setup and compact population outputs
from pathlib import Path
import sys
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import ipywidgets as widgets
from ipywidgets import HBox, Layout, VBox
from IPython.display import Markdown, clear_output, display


def find_repo_root(start=Path.cwd()):
    for candidate in [start.resolve(), *start.resolve().parents]:
        if (candidate / "src" / "population").is_dir():
            return candidate
    raise FileNotFoundError("Could not locate repository root")


ROOT = find_repo_root()
SRC = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

FEATURE_DIR = ROOT / "data" / "features" / "population"
FEATURE_CSV = FEATURE_DIR / "population_cv0_based_features.csv"
EVIDENCE_CSV = FEATURE_DIR / "population_cv0_template_evidence.csv"
TRACE_NPZ = FEATURE_DIR / "population_cv0_time_representations_v5.npz"
COLLAPSED_REGIME_CSV = FEATURE_DIR / "population_collapsed_regime_diagnostic.csv"
REGIME_EQUIVALENCE_CSV = FEATURE_DIR / "population_regime_response_equivalence.csv"

plt.rcParams.update({
    "figure.dpi": 120,
    "axes.spines.top": False,
    "axes.spines.right": False,
})

missing = [path for path in (FEATURE_CSV, EVIDENCE_CSV, TRACE_NPZ, COLLAPSED_REGIME_CSV, REGIME_EQUIVALENCE_CSV) if not path.is_file()]
if missing:
    raise FileNotFoundError(
        "Population feature outputs are missing. From the repository root, set "
        "PYTHONPATH to <repository>/src and run:\n"
        "python -m population.feature_engineering.feature_CLI run"
    )

features = pd.read_csv(FEATURE_CSV, low_memory=False)
evidence = pd.read_csv(EVIDENCE_CSV, low_memory=False)
collapsed_regimes = pd.read_csv(COLLAPSED_REGIME_CSV, low_memory=False)
regime_equivalence = pd.read_csv(REGIME_EQUIVALENCE_CSV, low_memory=False)
trace_archive = np.load(TRACE_NPZ, allow_pickle=False)
population_spike_counts_1ms = trace_archive["population_spike_counts_1ms"]
population_burst_occupancy_counts_10ms = trace_archive["population_burst_occupancy_counts_10ms"]

print(f"Features: {len(features):,} rows x {len(features.columns)} columns")
print(f"Evidence: {len(evidence):,} rows x {len(evidence.columns)} columns")
print(f"Collapsed-regime diagnostic: {len(collapsed_regimes):,} groups")
print(f"Response-equivalence assignments: {len(regime_equivalence):,} regime/frequency rows")
print(f"1 ms population traces: {population_spike_counts_1ms.shape} {population_spike_counts_1ms.dtype}")
print(f"10 ms population burst envelopes: {population_burst_occupancy_counts_10ms.shape} {population_burst_occupancy_counts_10ms.dtype}")

In [ ]:
# Coverage and integrity audit
TRIAL_KEYS = ["index_axon_id", "freq", "cv", "rep_index", "thresh_type"]
THRESH_ORDER = ["I_first", "I_volley5", "I_volley10", "I_5", "I_10", "I_15", "I_25", "I_50", "I_all"]
THRESH_LABELS = {
    "I_first": r"$I_{first}$", "I_volley5": r"$I_{volley5}$",
    "I_volley10": r"$I_{volley10}$", "I_5": r"$I_5$",
    "I_10": r"$I_{10}$", "I_15": r"$I_{15}$", "I_25": r"$I_{25}$",
    "I_50": r"$I_{50}$", "I_all": r"$I_{all}$",
}
EXPECTED_ROWS = 5 * 11 * 100 * 9

assert len(features) == EXPECTED_ROWS == len(evidence)
assert set(features["feature_version"]) == {"population-v5-complete-window-response-equivalence"}
assert not features.duplicated(TRIAL_KEYS).any()
for column in TRIAL_KEYS:
    assert np.array_equal(features[column].astype(str), evidence[column].astype(str))
assert set(features["n_population_axons"].astype(int)) == {420}
assert population_spike_counts_1ms.shape == (EXPECTED_ROWS, 2130)
assert population_burst_occupancy_counts_10ms.shape == (EXPECTED_ROWS, 213)
assert np.array_equal(trace_archive["trial_row"], np.arange(EXPECTED_ROWS))
assert np.allclose(trace_archive["cv"], features["cv"])
assert np.allclose(trace_archive["freq"], features["freq"])
assert np.array_equal(trace_archive["rep_index"], features["rep_index"])
assert np.array_equal(trace_archive["thresh_type"].astype(str), features["thresh_type"].astype(str))
assert np.allclose(features["active_axon_fraction"], features["active_axon_count"] / 420.0)
assert features["active_axon_count"].between(0, 420).all()
assert (features["total_population_spikes"] >= 0).all()
assert features["n_index_spikes"].notna().all()
assert np.array_equal(features["source_n_index_spikes"], features["n_index_spikes"])
assert np.array_equal(features["source_total_population_spikes"], features["total_population_spikes"])
assert np.array_equal(features["n_spikes_1ms"], features["total_population_spikes"])
assert features["n_events_10ms"].ge(0).all()
assert features["population_duty_cycle_10ms"].between(0, 1).all()
for suffix in ("1ms", "10ms"):
    assert features[f"silence_lv_degenerate_no_silence_{suffix}"].isin([True, False]).all()
    assert evidence[f"rate_endpoint_reliability_{suffix}"].between(0, 1).all()
    assert (evidence[f"rate_evidence_{suffix}"].abs() <= evidence[f"rate_endpoint_reliability_{suffix}"] + 1e-12).all()
for component in ("n_spikes_1ms", "n_events_10ms", "spikes_per_event_10ms"):
    assert set(evidence[f"rate_vote_{component}"].unique()).issubset({-1, 0, 1})
endpoint0 = np.isclose(evidence["cv"], 0.0)
endpoint1 = np.isclose(evidence["cv"], 1.0)
for suffix in ("1ms", "10ms"):
    assert np.allclose(evidence.loc[endpoint0, f"rate_evidence_{suffix}"], -evidence.loc[endpoint0, f"rate_endpoint_reliability_{suffix}"])
    assert np.allclose(evidence.loc[endpoint1, f"rate_evidence_{suffix}"], evidence.loc[endpoint1, f"rate_endpoint_reliability_{suffix}"])
for column in ["rate_evidence_1ms", "rate_evidence_10ms", "timing_evidence_1ms", "timing_evidence_10ms"]:
    assert evidence[column].between(-1, 1).all(), column

coverage = (features.groupby(["freq", "cv"], as_index=False)
            .agg(n_trials=("thresh_type", "size"),
                 mean_active_axons=("active_axon_count", "mean"),
                 mean_population_spikes=("total_population_spikes", "mean")))
display(Markdown("### Coverage and recruitment summary"))
display(coverage.head(15))
display(Markdown("### Endpoint reliability and response-equivalence summary"))
endpoint_reliability = (evidence[np.isclose(evidence["cv"], 0.0)]
                        .groupby(["freq", "thresh_type"])[["rate_endpoint_reliability_1ms", "rate_endpoint_reliability_10ms"]]
                        .mean().reset_index())
display(endpoint_reliability.head(18))
collapsed_summary = (collapsed_regimes.groupby(["freq", "collapsed_regimes"], as_index=False)
                     .agg(n_conditions=("rep_index", "size"), max_span_uA=("amp_span_uA", "max")))
display(collapsed_summary.head(20))
display(regime_equivalence[regime_equivalence["n_equivalent_regimes"] > 1].drop_duplicates(["freq", "equivalence_class_id"])[["freq", "canonical_regime", "member_regimes", "dropped_regimes", "observed_cv0_index_spikes", "nominal_targets_by_member"]])
print("Population feature, complete-window, evidence, and response-equivalence audits passed.")

In [ ]:
# Raw bounded evidence summaries shared by the dashboards
def sem_axis0(values):
    values = np.asarray(values, dtype=float)
    out = np.zeros(values.shape[1], dtype=float)
    for j in range(values.shape[1]):
        x = values[:, j]
        x = x[np.isfinite(x)]
        out[j] = 0.0 if x.size <= 1 else np.std(x, ddof=1) / np.sqrt(x.size)
    return out


def evidence_matrix(freq, thresh_type, column):
    subset = evidence[
        np.isclose(evidence["freq"], float(freq))
        & (evidence["thresh_type"] == str(thresh_type))
    ]
    cvs = np.sort(subset["cv"].unique().astype(float))
    values = subset[column].astype(float)
    defined_column = f"{column}_defined"
    if defined_column in subset:
        values = values.where(subset[defined_column].astype(bool), np.nan)
    plot_table = subset.assign(_plot_value=values)
    matrix = (plot_table.pivot(index="rep_index", columns="cv", values="_plot_value")
              .reindex(columns=cvs).sort_index().to_numpy(float))
    return cvs, matrix


def mean_evidence(freq, thresh_type, rate_column, timing_column):
    cvs, rate = evidence_matrix(freq, thresh_type, rate_column)
    cvs_t, timing = evidence_matrix(freq, thresh_type, timing_column)
    assert np.allclose(cvs, cvs_t)
    return {
        "cvs": cvs,
        "rate_mean": np.nanmean(rate, axis=0),
        "rate_sem": sem_axis0(rate),
        "timing_mean": np.nanmean(timing, axis=0),
        "timing_sem": sem_axis0(timing),
    }


def selected_rows(freq, thresh_type, rep):
    subset = features[
        np.isclose(features["freq"], float(freq))
        & (features["thresh_type"] == str(thresh_type))
        & (features["rep_index"] == int(rep))
    ].sort_values("cv")
    if len(subset) != 11:
        raise RuntimeError(f"Expected 11 CV rows, found {len(subset)}")
    return subset


print("Dashboard helpers ready.")

In [ ]:
# Selected-condition workflow: population analogue of the single-axon dashboard
selected_output = widgets.Output()
frequency_options = sorted(features["freq"].unique().astype(float))
regime_options = [x for x in THRESH_ORDER if x in set(features["thresh_type"])]
rep_options = sorted(features["rep_index"].unique().astype(int))

frequency_widget = widgets.Dropdown(options=frequency_options, value=200.0, description="freq (Hz)", layout=Layout(width="180px"))
regime_widget = widgets.Dropdown(options=regime_options, value="I_10", description="regime", layout=Layout(width="210px"))
rep_widget = widgets.Dropdown(options=rep_options, value=0, description="rep", layout=Layout(width="170px"))
envelope_widget = widgets.Dropdown(
    options=[
        ("Binary timing support (primary)", "binary"),
        ("Recruited-axon fraction (intensity)", "fraction"),
        ("Concurrent axon count (intensity)", "count"),
    ],
    value="binary",
    description="envelope",
    layout=Layout(width="330px"),
)


def render_selected(*_):
    with selected_output:
        clear_output(wait=True)
        freq = float(frequency_widget.value)
        regime = str(regime_widget.value)
        rep = int(rep_widget.value)
        envelope_mode = str(envelope_widget.value)
        rows = selected_rows(freq, regime, rep)
        cvs = rows["cv"].to_numpy(float)
        row_indices = rows.index.to_numpy(int)
        spike_traces = population_spike_counts_1ms[row_indices]
        burst_traces = population_burst_occupancy_counts_10ms[row_indices]
        colors = plt.cm.tab10(np.linspace(0, 1, len(cvs)))

        evidence_rows = evidence.loc[row_indices]
        summary1 = mean_evidence(freq, regime, "rate_evidence_1ms", "timing_evidence_1ms")
        summary10 = mean_evidence(freq, regime, "rate_evidence_10ms", "timing_evidence_10ms")

        fig = plt.figure(figsize=(18, 16), constrained_layout=True)
        grid = fig.add_gridspec(5, 2, height_ratios=[1.45, 1.15, 1.0, 1.0, 1.0])
        ax_raster = fig.add_subplot(grid[0, :])
        ax_envelope = fig.add_subplot(grid[1, :], sharex=ax_raster)
        ax_feature_rate = fig.add_subplot(grid[2, 0])
        ax_feature_timing = fig.add_subplot(grid[2, 1])
        ax_evidence_rate = fig.add_subplot(grid[3, 0])
        ax_evidence_timing = fig.add_subplot(grid[3, 1])
        ax_mean_rate = fig.add_subplot(grid[4, 0])
        ax_mean_timing = fig.add_subplot(grid[4, 1])

        # Each vertical mark is an occupied 1 ms population bin; width scales with spike count.
        global_spike_max = max(1.0, float(np.max(spike_traces)))
        for y, (cv, counts, color) in enumerate(zip(cvs, spike_traces, colors)):
            occupied = np.flatnonzero(counts)
            if occupied.size:
                widths = 0.35 + 1.8 * np.sqrt(counts[occupied] / global_spike_max)
                ax_raster.vlines(occupied + 0.5, y - 0.32, y + 0.32, color=color, linewidth=widths, alpha=0.85)
        ax_raster.set_yticks(np.arange(len(cvs)), [f"CV={cv:g}" for cv in cvs])
        ax_raster.set_ylim(len(cvs) - 0.45, -0.55)
        ax_raster.set_xlim(0, 2000)
        ax_raster.set_ylabel("Population spike density")
        ax_raster.set_title("Population spike-count raster (1 ms bins; mark width encodes spike count)")

        # Each axon contributes one unit while its variable-duration burst occupies a 10 ms bin.
        # Binary support is the primary timing representation; magnitude is intensity only.
        global_burst_max = max(1.0, float(np.max(burst_traces)))
        time10 = np.arange(burst_traces.shape[1] + 1, dtype=float) * 10.0
        for y, (cv, counts, active_axons, color) in enumerate(zip(cvs, burst_traces, rows["active_axon_count"], colors)):
            if envelope_mode == "fraction":
                amplitude = counts / max(1.0, float(active_axons))
            elif envelope_mode == "binary":
                amplitude = (counts > 0).astype(float)
            else:
                amplitude = counts / global_burst_max
            trace = y - 0.72 * np.r_[amplitude, amplitude[-1]]
            ax_envelope.step(time10, trace, where="post", color=color, linewidth=1.5)
        ax_envelope.set_yticks(np.arange(len(cvs)), [f"CV={cv:g}" for cv in cvs])
        ax_envelope.set_ylim(len(cvs) - 0.45, -1.0)
        ax_envelope.set_xlim(0, 2000)
        ax_envelope.set_xlabel("Time (ms)")
        envelope_labels = {
            "fraction": ("CV / active fraction", "Population burst envelope intensity (fraction of recruited axons bursting)"),
            "binary": ("CV / binary timing", "Population burst timing envelope (binary support; variable burst duration)"),
            "count": ("CV / active count", f"Population burst envelope intensity (concurrent axons; panel max={global_burst_max:g})"),
        }
        envelope_ylabel, envelope_title = envelope_labels[envelope_mode]
        ax_envelope.set_ylabel(envelope_ylabel)
        ax_envelope.set_title(envelope_title)

        ax_feature_rate.plot(cvs, rows["n_spikes_1ms"], marker="o", label="n_spikes (1 ms population)")
        ax_feature_rate.plot(cvs, rows["n_events_10ms"], marker="o", label="n_events (10 ms population bursts)")
        rate_right = ax_feature_rate.twinx()
        rate_right.plot(cvs, rows["spikes_per_event_10ms"], marker="o", linestyle="--", color="C2", label="spikes / population event")
        ax_feature_rate.set_title("FEATURES (selected rep) - RATE")
        ax_feature_rate.set_xlabel("CV")
        ax_feature_rate.set_ylabel("Population count")
        rate_right.set_ylabel("Spikes per population event")
        handles1, labels1 = ax_feature_rate.get_legend_handles_labels()
        handles2, labels2 = rate_right.get_legend_handles_labels()
        ax_feature_rate.legend(handles1 + handles2, labels1 + labels2, frameon=False, fontsize=8, loc="upper right")

        ax_feature_timing.plot(cvs, rows["silence_lv_1ms"], marker="o", linestyle="--", label="silence_lv (1 ms population)")
        ax_feature_timing.plot(cvs, rows["silence_lv_10ms"], marker="o", label="silence_lv (10 ms population bursts)")
        ax_feature_timing.set_title("FEATURES (selected rep) - TIMING")
        ax_feature_timing.set_xlabel("CV")
        ax_feature_timing.legend(frameon=False, fontsize=8)

        reliability1 = float(evidence_rows["rate_endpoint_reliability_1ms"].iloc[0])
        reliability10 = float(evidence_rows["rate_endpoint_reliability_10ms"].iloc[0])
        ax_evidence_rate.plot(cvs, evidence_rows["rate_evidence_1ms"], marker="o", linestyle="--", linewidth=2.0, label=f"combined 1 ms (R={reliability1:.2f})")
        ax_evidence_rate.plot(cvs, evidence_rows["rate_evidence_10ms"], marker="o", linewidth=2.0, label=f"combined 10 ms (R={reliability10:.2f})")
        ax_evidence_rate.plot(cvs, evidence_rows["rate_vote_n_events_10ms"] / 2.0, linestyle=":", alpha=0.8, label="10 ms event-count contribution")
        ax_evidence_rate.plot(cvs, evidence_rows["rate_vote_spikes_per_event_10ms"] / 2.0, linestyle=":", alpha=0.8, label="10 ms spikes/event contribution")
        ax_evidence_rate.axhline(0, color="C0", alpha=0.4)
        ax_evidence_rate.set_title("EVIDENCE (selected rep) - endpoint-gated RATE hard-JND (k=2.5)")
        ax_evidence_rate.set_xlabel("CV")
        ax_evidence_rate.set_ylabel("E_rate (raw)")
        ax_evidence_rate.legend(frameon=False, fontsize=8)

        timing1 = evidence_rows["timing_evidence_1ms"].where(evidence_rows["timing_evidence_1ms_defined"].astype(bool), np.nan)
        timing10 = evidence_rows["timing_evidence_10ms"].where(evidence_rows["timing_evidence_10ms_defined"].astype(bool), np.nan)
        delta1 = float(evidence_rows["timing_endpoint_abs_delta_1ms"].iloc[0])
        delta10 = float(evidence_rows["timing_endpoint_abs_delta_10ms"].iloc[0])
        ax_evidence_timing.plot(cvs, timing1, marker="o", linestyle="--", label=f"timing 1 ms (endpoint delta={delta1:.3g})")
        ax_evidence_timing.plot(cvs, timing10, marker="o", label=f"timing 10 ms (endpoint delta={delta10:.3g})")
        ax_evidence_timing.axhline(0, color="C0", alpha=0.4)
        ax_evidence_timing.set_title("EVIDENCE (selected rep) - TIMING")
        ax_evidence_timing.set_xlabel("CV")
        ax_evidence_timing.set_ylabel("E_time (raw)")
        ax_evidence_timing.legend(frameon=False, fontsize=8)

        for summary, label, style in [(summary1, "bin=1 ms", "--"), (summary10, "bin=10 ms", "-")]:
            rate_mean = np.asarray(summary["rate_mean"], float)
            rate_lo = rate_mean - np.asarray(summary["rate_sem"], float)
            rate_hi = rate_mean + np.asarray(summary["rate_sem"], float)
            timing_mean = np.asarray(summary["timing_mean"], float)
            timing_lo = timing_mean - np.asarray(summary["timing_sem"], float)
            timing_hi = timing_mean + np.asarray(summary["timing_sem"], float)
            ax_mean_rate.plot(cvs, rate_mean, marker="o", linestyle=style, label=label)
            ax_mean_rate.fill_between(cvs, rate_lo, rate_hi, alpha=0.15)
            ax_mean_timing.plot(cvs, timing_mean, marker="o", linestyle=style, label=label)
            ax_mean_timing.fill_between(cvs, timing_lo, timing_hi, alpha=0.15)
        for ax in (ax_mean_rate, ax_mean_timing):
            ax.axhline(0, color="C0", alpha=0.4)
            ax.set_xlabel("CV")
            ax.legend(frameon=False, fontsize=8)
            ax.set_ylim(-1.05, 1.05)
        ax_mean_rate.set_title("MEAN +/- SEM evidence across reps - RATE (endpoint-gated hard-JND)")
        ax_mean_rate.set_ylabel("mean E_rate")
        ax_mean_timing.set_title("MEAN +/- SEM evidence across reps - TIMING (bounded signed distance)")
        ax_mean_timing.set_ylabel("mean E_time")

        for ax in (ax_feature_rate, ax_feature_timing, ax_evidence_rate, ax_evidence_timing, ax_mean_rate, ax_mean_timing):
            ax.grid(alpha=0.2)
        fig.suptitle(
            f"population discrimination_core-consistent workflow | freq={freq:g} Hz | "
            f"regime={regime} | rep={rep} | bins=(1,10) ms | endpoint-gated JND k=2.5",
            fontsize=13,
        )
        plt.show()


for control in (frequency_widget, regime_widget, rep_widget, envelope_widget):
    control.observe(render_selected, names="value")
display(VBox([HBox([frequency_widget, regime_widget, rep_widget]), envelope_widget, selected_output]))
render_selected()

In [ ]:
# All index-regime population evidence curves (parallel to the single-axon 2 x 2 figure)
all_evidence_output = widgets.Output()
all_evidence_frequency = widgets.Dropdown(options=frequency_options, value=200.0, description="freq (Hz)", layout=Layout(width="180px"))


def render_all_evidence(*_):
    with all_evidence_output:
        clear_output(wait=True)
        freq = float(all_evidence_frequency.value)
        fig, axes = plt.subplots(2, 2, figsize=(15, 9), sharex=True, sharey=True)
        panels = [
            (axes[0, 0], "rate_mean", "rate_sem", "RATE evidence (bin=1 ms)"),
            (axes[0, 1], "timing_mean", "timing_sem", "TIMING evidence (bin=1 ms)"),
            (axes[1, 0], "rate_mean", "rate_sem", "RATE evidence (bin=10 ms)"),
            (axes[1, 1], "timing_mean", "timing_sem", "TIMING evidence (bin=10 ms)"),
        ]
        colors = plt.cm.tab10(np.linspace(0, 1, len(THRESH_ORDER)))
        for regime, color in zip(THRESH_ORDER, colors):
            summary1 = mean_evidence(freq, regime, "rate_evidence_1ms", "timing_evidence_1ms")
            summary10 = mean_evidence(freq, regime, "rate_evidence_10ms", "timing_evidence_10ms")
            for ax, summary, mean_key, sem_key in [
                (axes[0, 0], summary1, "rate_mean", "rate_sem"),
                (axes[0, 1], summary1, "timing_mean", "timing_sem"),
                (axes[1, 0], summary10, "rate_mean", "rate_sem"),
                (axes[1, 1], summary10, "timing_mean", "timing_sem"),
            ]:
                mean = np.asarray(summary[mean_key], float)
                lo = mean - np.asarray(summary[sem_key], float)
                hi = mean + np.asarray(summary[sem_key], float)
                ax.plot(summary["cvs"], mean, marker="o", linewidth=1.8, color=color, label=THRESH_LABELS[regime])
                ax.fill_between(summary["cvs"], lo, hi, color=color, alpha=0.08)
        for ax, _, _, title in panels:
            ax.axhline(0, color="C0", alpha=0.45)
            ax.set_title(title)
            ax.set_xlabel("CV")
            ax.set_ylabel("Bounded evidence")
            ax.grid(alpha=0.2)
            ax.set_ylim(-1.05, 1.05)
        handles, labels = axes[0, 0].get_legend_handles_labels()
        fig.legend(handles, labels, loc="lower center", ncol=5, frameon=False, bbox_to_anchor=(0.5, -0.01))
        fig.suptitle(
            f"ALL index regimes | population | freq={freq:g} Hz | "
            "endpoint-gated hard-JND rate + bounded signed-distance timing | JND k=2.5"
        )
        fig.tight_layout(rect=(0, 0.07, 1, 0.96))
        plt.show()


all_evidence_frequency.observe(render_all_evidence, names="value")
display(VBox([all_evidence_frequency, all_evidence_output]))
render_all_evidence()

In [ ]:
# Population-specific feature curves across all index regimes
all_features_output = widgets.Output()
all_features_frequency = widgets.Dropdown(options=frequency_options, value=200.0, description="freq (Hz)", layout=Layout(width="180px"))


def render_all_features(*_):
    with all_features_output:
        clear_output(wait=True)
        freq = float(all_features_frequency.value)
        subset = features[np.isclose(features["freq"], freq)].copy()
        fig, axes = plt.subplots(2, 2, figsize=(15, 9), sharex=True)
        specs = [
            ("n_spikes_1ms", "Mean n_spikes (1 ms population)"),
            ("n_events_10ms", "Mean n_events (10 ms population bursts)"),
            ("spikes_per_event_10ms", "Mean spikes per population event (10 ms)"),
            ("silence_lv_10ms", "Mean population silence_lv (10 ms)"),
        ]
        colors = plt.cm.tab10(np.linspace(0, 1, len(THRESH_ORDER)))
        for regime, color in zip(THRESH_ORDER, colors):
            regime_rows = subset[subset["thresh_type"] == regime]
            for ax, (column, _) in zip(axes.flat, specs):
                grouped = regime_rows.groupby("cv")[column]
                mean = grouped.mean()
                sem = grouped.sem()
                ax.plot(mean.index, mean.values, marker="o", linewidth=1.8, color=color, label=THRESH_LABELS[regime])
                ax.fill_between(mean.index, mean.values - sem.values, mean.values + sem.values, color=color, alpha=0.08)
        for ax, (_, title) in zip(axes.flat, specs):
            ax.set_title(title)
            ax.set_xlabel("CV")
            ax.grid(alpha=0.2)
        handles, labels = axes[0, 0].get_legend_handles_labels()
        fig.legend(handles, labels, loc="lower center", ncol=5, frameon=False, bbox_to_anchor=(0.5, -0.01))
        fig.suptitle(f"Population features across all index regimes | freq={freq:g} Hz")
        fig.tight_layout(rect=(0, 0.07, 1, 0.96))
        plt.show()


all_features_frequency.observe(render_all_features, names="value")
display(VBox([all_features_frequency, all_features_output]))
render_all_features()

In [ ]:
# Observer-facing diagnostics retained for the next stage
endpoint_rows = evidence[np.isclose(evidence["cv"], 0.0)].copy()

rate_diagnostic = (endpoint_rows.groupby(["freq", "thresh_type"], as_index=False)
                   .agg(rate_endpoint_reliability_1ms=("rate_endpoint_reliability_1ms", "mean"),
                        rate_endpoint_reliability_10ms=("rate_endpoint_reliability_10ms", "mean"),
                        timing_endpoint_abs_delta_1ms=("timing_endpoint_abs_delta_1ms", "median"),
                        timing_endpoint_abs_delta_10ms=("timing_endpoint_abs_delta_10ms", "median"),
                        timing_defined_fraction_1ms=("timing_evidence_1ms_defined", "mean"),
                        timing_defined_fraction_10ms=("timing_evidence_10ms_defined", "mean"),
                        cv0_no_silence_fraction_10ms=("timing_template_cv0_no_silence_10ms", "mean"),
                        cv1_no_silence_fraction_10ms=("timing_template_cv1_no_silence_10ms", "mean")))

display(Markdown("### Endpoint separation and cue availability"))
display(rate_diagnostic)
display(Markdown("### Amplitude proximity diagnostic (not used to merge regimes)"))
display(collapsed_regimes.head(50))
display(Markdown("### Frequency-wide response-equivalence classes used by the observer"))
display(regime_equivalence[regime_equivalence["n_equivalent_regimes"] > 1].drop_duplicates(["freq", "equivalence_class_id"]))
print(
    "Amplitude proximity is descriptive only. The observer uses the full-CV x replicate "
    "response-equivalence map below, so pattern-specific coincidences are not merged."
)
